Lab 8 SVM

Robert Armenta, Faith Yang (Mon-Wed Class)

Dataset Number 4 Ovo vs Ova: https://www.kaggle.com/datasets/olcaybolat1/dermatology-dataset-classification

In [ ]:
# Ova vs Ov0 Dataset

# Import a built-in dataset (originally used for testing multi-class classification like Iris)
from sklearn import datasets
# Split data into training and testing sets
from sklearn.model_selection import train_test_split
# Support Vector Machine (SVM) classifier
from sklearn.svm import SVC
# Multi-class strategies used when there are more than 2 classe
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
# Metrics to evaluate how well the model performs
# classification_report = precision, recall, F1-score breakdown
from sklearn.metrics import accuracy_score, classification_report
# Used to create graphs/plots for visualization
import matplotlib.pyplot as plt
# Handles numerical operations like arrays and math functions
import numpy as np
# Used to load and manage datasets (especially CSV files)
import pandas as pd
# Used to measure how long the model takes to train
import time
from matplotlib.lines import Line2D


# Print a header to make output easier to read in the console
print("=" * 70)
print("MULTI-CLASS CLASSIFICATION STRATEGY COMPARISON")
print("=" * 70)

In [ ]:
# Load dataset from CSV file
# Upload the dataset file from your computer
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("dermatology_database_1.csv")

print("\nDataset Loaded Successfully")
print(f"Shape: {df.shape}")

# Preview first few rows
df.head()

In [ ]:
# CLEAN DATA
# Replace '?' with NaN
df.replace("?", np.nan, inplace=True)

# Convert all columns to numeric
df = df.apply(pd.to_numeric)

# Fill missing values (age column usually)
df.fillna(df.median(), inplace=True)

In [ ]:
#FEATURES & TARGET
X = df.drop("class", axis=1)
y = df["class"]

In [ ]:

print("\nDataset: Dermatology")
print("Samples:", len(y))
print("Features:", X.shape[1])
print("Classes:", len(np.unique(y)))
print("Class labels:", sorted(np.unique(y)))

In [ ]:
# Split data into training (70%) and testing (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("\n" + "=" * 70)
print("STRATEGY 1: Standard SVM")
print("=" * 70)

# Simple explanation
print("Uses default SVM")
print("Automatically compares classes in pairs (One-vs-One)")
print("Example: Class 0 vs 1, 0 vs 2, 1 vs 2")

In [ ]:
# Start timer
start_time = time.time()

# Create and train SVM model
model_standard = SVC(kernel='rbf')
model_standard.fit(X_train, y_train)

# Calculate training time
train_time_standard = time.time() - start_time

# Make predictions on test data
y_pred_standard = model_standard.predict(X_test)

# Calculate accuracy
accuracy_standard = accuracy_score(y_test, y_pred_standard)

# Print results
print("\nResults:")
print("Training time:", round(train_time_standard, 4), "seconds")
print("Accuracy:", round(accuracy_standard * 100, 2), "%")

In [ ]:
print("\n" + "=" * 70)
print("STRATEGY 2: One-vs-Rest (OvA)")
print("=" * 70)
print("Description:")
print("  - Also called One-vs-All")
print("  - For each class, trains ONE classifier against all other classes")
print("  - For 3 classes: creates 3 binary classifiers")
print("  - Classifiers: Class0 vs (Class1+Class2), Class1 vs (Class0+Class2),")
print("                 Class2 vs (Class0+Class1)")
print("  - Prediction: Choose the class with highest confidence score")

In [ ]:
# Start timer
start_time = time.time()

# Create and train One-vs-Rest (OvR) model
model_ovr = OneVsRestClassifier(SVC(kernel='rbf'))
model_ovr.fit(X_train, y_train)

# Calculate training time
train_time_ovr = time.time() - start_time

# Make predictions
y_pred_ovr = model_ovr.predict(X_test)

# Calculate accuracy
accuracy_ovr = accuracy_score(y_test, y_pred_ovr)

# Print results
print("\nResults:")
print("Training time:", round(train_time_ovr, 4), "seconds")
print("Accuracy:", round(accuracy_ovr * 100, 2), "%")

# Number of classifiers created (one per class)
print("Classifiers trained:", len(model_ovr.estimators_))

In [ ]:
print("\n" + "=" * 70)
print("STRATEGY 3: One-vs-One (OvO)")
print("=" * 70)

# Simple explanation
print("Trains a model for every pair of classes")
print("Example: Class 0 vs 1, 0 vs 2, 1 vs 2")
print("Final prediction is based on voting")

# Start timer
start_time = time.time()

# Create and train OvO model
model_ovo = OneVsOneClassifier(SVC(kernel='rbf'))
model_ovo.fit(X_train, y_train)

# Calculate training time
train_time_ovo = time.time() - start_time

# Make predictions
y_pred_ovo = model_ovo.predict(X_test)

# Calculate accuracy
accuracy_ovo = accuracy_score(y_test, y_pred_ovo)

# Print results
print("\nResults:")
print("Training time:", round(train_time_ovo, 4), "seconds")
print("Accuracy:", round(accuracy_ovo * 100, 2), "%")

# Number of classifiers created
print("Classifiers trained:", len(model_ovo.estimators_))

In [ ]:
print("\n" + "=" * 70)
print("FINAL COMPARISON SUMMARY")
print("=" * 70)

# Print comparison table
print("\nStrategy        Accuracy     Time        Classifiers")
print("-" * 60)

print("Standard SVM   ", round(accuracy_standard * 100, 2), "%  ",
      round(train_time_standard, 4), "s   N/A")

print("OvR            ", round(accuracy_ovr * 100, 2), "%  ",
      round(train_time_ovr, 4), "s  ", len(model_ovr.estimators_))

print("OvO            ", round(accuracy_ovo * 100, 2), "%  ",
      round(train_time_ovo, 4), "s  ", len(model_ovo.estimators_))


# Store accuracies to compare
accuracies = {
    "Standard SVM": accuracy_standard,
    "OvR": accuracy_ovr,
    "OvO": accuracy_ovo
}

# Find best strategy
best_strategy = max(accuracies, key=accuracies.get)

print("\nBest accuracy:", best_strategy)


# Section header for next part
print("\n" + "=" * 70)
print("DETAILED REPORT (OvO)")
print("=" * 70)

print("\nGenerating visualization...")

In [ ]:
# Use first 2 features from dermatology data for plotting
X_2d = X.iloc[:, :2].values
y_2d = y.values

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y_2d, test_size=0.3, random_state=42, stratify=y_2d
)

models_2d = {
    "Standard SVM": SVC(kernel='rbf'),
    "OvR": OneVsRestClassifier(SVC(kernel='rbf')),
    "OvO": OneVsOneClassifier(SVC(kernel='rbf'))
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

# Loop through models
for idx, (name, model) in enumerate(models_2d.items()):

    # Train model
    model.fit(X_train_2d, y_train_2d)

    # Test accuracy
    y_pred_2d = model.predict(X_test_2d)
    acc_2d = accuracy_score(y_test_2d, y_pred_2d)

    # Predict for graph
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot decision boundary
    axes[idx].contourf(xx, yy, Z, alpha=0.4)

    # Plot training points
    axes[idx].scatter(X_train_2d[:, 0], X_train_2d[:, 1],
                      c=y_train_2d, edgecolors='k')

    # Plot test points (squares)
    axes[idx].scatter(X_test_2d[:, 0], X_test_2d[:, 1],
                      c=y_test_2d, edgecolors='red', marker='s')

    # Labels
    axes[idx].set_xlabel("Feature 1")
    axes[idx].set_ylabel("Feature 2")
    axes[idx].set_title(f"{name} (Acc: {round(acc_2d*100,1)}%)")


    # Create custom legend (circle = train, square = test)
from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='gray', label='Training data'),

    Line2D([0], [0], marker='s', color='w',
           markerfacecolor='gray', markeredgecolor='red',
           label='Test data')
]

# Add legend to middle plot
axes[1].legend(handles=legend_elements, loc='upper left')

# Title for entire figure
plt.suptitle("Model Comparison (Train vs Test Data)")

# Fix layout and show plot
plt.tight_layout()
plt.show()


In [ ]:
for n_classes in [3, 5, 10, 20, 50, 100]:
    n_ovr = n_classes
    n_ovo = n_classes * (n_classes - 1) // 2
    print(f"{n_classes:<12} {n_ovr:<20} {n_ovo}")


**Lab Report**

The analysis used a dermatology dataset to compare two common multi-class classification approaches—One-vs-Rest (OvA) and One-vs-One (OvO) using a Support Vector Machine (SVM) model. The data was first cleaned by converting all values to numeric format and filling in missing values with median estimates to maintain consistency. The dataset was then split into training (70%) and testing (30%) sets to evaluate real-world performance. Both models were trained and tested on the same data, and their performance was measured using accuracy along with precision, recall, and F1-score to assess classification quality across all categories. The results show how each method handles multi-class problems, with OvO typically providing slightly better accuracy at the cost of longer training time due to more model comparisons, while OvA is faster and more efficient but may be slightly less precise in some cases. Overall, the project demonstrates a practical comparison of classification strategies and highlights the trade-off between speed and accuracy when selecting a model for real-world applications.